# 概述
管理器模式是一个多代理体系结构，其中中央管理器代理协调专门的工作者代理。当任务需要不同类型的专业知识时，这种方法非常有效。您不是构建一个代理来管理跨领域的工具选择，而是创建由了解整个工作流程的主管协调的专注的专家。

在本教程中，您将构建一个个人助理系统，通过一个现实的工作流程来展示这些好处。该系统将协调两名职责完全不同的专家：

处理日程安排、可用性检查和事件管理的日历代理。

管理通信、起草消息和发送通知的电子邮件代理。

我们还将纳入人工循环审查，以允许用户根据需要批准、编辑和拒绝操作（例如出站电子邮件）。

多代理体系结构允许您跨工作者对工具进行分区，每个工作者都有自己的提示或指令。考虑一个可以直接访问所有日历和电子邮件API的代理：它必须从许多类似的工具中进行选择，理解每个API的确切格式，并同时处理多个域。如果性能下降，将相关工具和相关提示分离到逻辑组中（部分是为了管理迭代改进）可能会有所帮助。
# human-in-the-loop
人在循环（human -in- loop， HITL）中间件允许您向代理工具调用添加人工监督。当模型提出可能需要审查的操作时（例如，写入文件或执行SQL），中间件可以暂停执行并等待决策。

它通过根据可配置策略检查每个工具调用来实现这一点。如果需要干预，中间件会发出中断，停止执行。使用LangGraph的持久层保存图形状态，因此可以安全地暂停执行并稍后继续执行。

然后，人工决策决定接下来会发生什么：可以按原样批准操作（批准），在运行前修改操作（编辑），或者通过反馈拒绝操作（拒绝）。
## 配置中断
要使用HITL，在创建代理时将中间件添加到代理的中间件列表中。

您可以使用工具操作到每个操作所允许的决策类型的映射来配置它。当工具调用与映射中的操作匹配时，中间件将中断执行。


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 


agent = create_agent(
    model="gpt-4o",
    tools=[write_file_tool, execute_sql_tool, read_data_tool],
    middleware=[
        HumanInTheLoopMiddleware( 
            interrupt_on={
                "write_file": True,  # All decisions (approve, edit, reject) allowed
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},  # No editing allowed
                # Safe operation, no approval needed
                "read_data": False,
            },
            # Prefix for interrupt messages - combined with tool name and args to form the full message
            # e.g., "Tool execution pending approval: execute_sql with query='DELETE FROM...'"
            # Individual tools can override this by specifying a "description" in their interrupt config
            description_prefix="Tool execution pending approval",
        ),
    ],
    # Human-in-the-loop requires checkpointing to handle interrupts.
    # In production, use a persistent checkpointer like AsyncPostgresSaver.
    checkpointer=InMemorySaver(),  
)

您必须配置一个检查指针，以便跨中断保持图形状态。在生产环境中，使用像AsyncPostgresSaver这样的持久检查指针。对于测试或原型，使用InMemorySaver。

在调用代理时，传递一个包含线程ID的配置，以便将执行与会话线程关联起来。有关详细信息，请参阅LangGraph中断文档。
## 回应中断
调用代理时，它会一直运行，直到完成或引发中断。当工具调用匹配您在interrupt_on中配置的策略时，将触发中断。在这种情况下，调用结果将包含一个__interrupt__字段，其中包含需要检查的操作。然后，您可以将这些操作呈现给审阅者，并在提供决策后继续执行。

In [ ]:
from langgraph.types import Command

# Human-in-the-loop leverages LangGraph's persistence layer.
# You must provide a thread ID to associate the execution with a conversation thread,
# so the conversation can be paused and resumed (as is needed for human review).
config = {"configurable": {"thread_id": "some_id"}} 
# Run the graph until the interrupt is hit.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete old records from the database",
            }
        ]
    },
    config=config 
)

# The interrupt contains the full HITL request with action_requests and review_configs
print(result['__interrupt__'])
# > [
# >    Interrupt(
# >       value={
# >          'action_requests': [
# >             {
# >                'name': 'execute_sql',
# >                'arguments': {'query': 'DELETE FROM records WHERE created_at < NOW() - INTERVAL \'30 days\';'},
# >                'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {...}'
# >             }
# >          ],
# >          'review_configs': [
# >             {
# >                'action_name': 'execute_sql',
# >                'allowed_decisions': ['approve', 'reject']
# >             }
# >          ]
# >       }
# >    )
# > ]


# Resume with approval decision
agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "edit", "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

## 决定种类


In [ ]:
agent.invoke(
    Command(
        # Decisions are provided as a list, one per action under review.
        # The order of decisions must match the order of actions
        # listed in the `__interrupt__` request.
        resume={
            "decisions": [
                {
                    "type": "approve",
                }
            ]
        }
    ),
    config=config  # Same thread ID to resume the paused conversation
)

In [ ]:
agent.invoke(
    Command(
        # Decisions are provided as a list, one per action under review.
        # The order of decisions must match the order of actions
        # listed in the `__interrupt__` request.
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "new_tool_name",
                        # Arguments to pass to the tool.
                        "args": {"key1": "new_value", "key2": "original_value"},
                    }
                }
            ]
        }
    ),
    config=config  # Same thread ID to resume the paused conversation
)

In [ ]:
agent.invoke(
    Command(
        # Decisions are provided as a list, one per action under review.
        # The order of decisions must match the order of actions
        # listed in the `__interrupt__` request.
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation about why the action was rejected
                    "message": "No, this is wrong because ..., instead do this ...",
                }
            ]
        }
    ),
    config=config  # Same thread ID to resume the paused conversation
)

## 多决策
当多个动作被审查时，为每个动作提供一个决定，顺序与它们在中断中出现的顺序相同：

In [ ]:
{
    "decisions": [
        {"type": "approve"},
        {
            "type": "edit",
            "edited_action": {
                "name": "tool_name",
                "args": {"param": "new_value"}
            }
        },
        {
            "type": "reject",
            "message": "This action is not allowed"
        }
    ]
}

## 执行声明周期
中间件定义了一个after_model钩子，该钩子在模型生成响应之后，但在执行任何工具调用之前运行：
1. 代理调用模型来生成响应。
2. 中间件检查工具调用的响应。
3. 如果任何调用需要人工输入，中间件将用action_requests和review_configs构建一个HITLRequest并调用interrupt。
4. 代理等待人类的决定。
5. 基于HITLResponse决策，中间件执行已批准或已编辑的调用，为被拒绝的调用合成ToolMessage，并继续执行。
# 具体实现
首先，初始化聊天模型

In [1]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import os
model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-plus",  # 此处以qwen-plus为例，您可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/getting-started/models
    # other params...
)

## 定义工具
首先定义需要结构化输入的工具。在实际的应用程序中，这些将调用实际的api（谷歌Calendar、SendGrid等）。在本教程中，您将使用存根来演示该模式。

In [2]:
from langchain_core.tools import tool

@tool
def create_calendar_event(
    title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"


@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"


@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]

## 创建特定的子agent
### 创建日历agent
日历agent理解自然语言调度请求，并将其转换为精确的API调用。它处理日期解析、可用性检查和事件创建。

In [3]:
from langchain.agents import create_agent


CALENDAR_AGENT_PROMPT = (
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
)

测试：

In [4]:
query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_available_time_slots (call_7dfe45090ee3451785b25d)
 Call ID: call_7dfe45090ee3451785b25d
  Args:
    attendees: []
    date: 2023-04-18
    duration_minutes: 60
================================= Tool Message =================================
Name: get_available_time_slots

["09:00", "14:00", "16:00"]
================================== Ai Message ==================================
Tool Calls:
  create_calendar_event (call_e49b49ce04d64b1085049c)
 Call ID: call_e49b49ce04d64b1085049c
  Args:
    title: Team Meeting
    start_time: 2023-04-18T14:00:00
    end_time: 2023-04-18T15:00:00
    attendees: []
    location:
================================= Tool Message =================================
Name: create_calendar_event

Event created: Team Meeting from 2023-04-18T14:00:00 to 2023-04-18T15:00:00 with 0 attendees
================================== Ai Message ===============================

代理将“next Tuesday at 2pm”解析为ISO格式（“2024-01-16T14:00:00”），计算结束时间，调用create_calendar_event，并返回自然语言确认。

## 创建电子邮件agent
电子邮件代理处理消息组合和发送。它侧重于提取收件人信息，制作合适的主题行和正文，以及管理电子邮件通信。

In [6]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

测试

In [7]:
query = "Send the design team a reminder about reviewing the new mockups"

for step in email_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  send_email (call_9db1dff1cb164fd3a27294)
 Call ID: call_9db1dff1cb164fd3a27294
  Args:
    to: ['design.team@company.com']
    subject: Reminder: Review New Mockups
    body: Hi Team,

Just a quick reminder to review the new mockups when you get a chance. Your feedback is important to ensure we're aligned before moving forward.

Thanks!
[Your Name]
================================= Tool Message =================================
Name: send_email

Email sent to design.team@company.com - Subject: Reminder: Review New Mockups
================================== Ai Message ==================================

I've sent a reminder email to the design team about reviewing the new mockups. The message was delivered to design.team@company.com with the subject "Reminder: Review New Mockups."


代理从非正式请求中推断接收者，制作专业的主题行和正文，调用send_email，并返回确认。每个子代理都有一个特定于领域的工具和提示，允许它在特定的任务中表现出色。
## 将子agent包装为工具
现在将每个子代理包装为主管程序可以调用的工具。这是创建分层系统的关键架构步骤。主管将看到高级工具，如“schedule_event”，而不是低级工具，如“create_calendar_event”。

In [9]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text

工具描述可以帮助主管决定何时使用每个工具，因此要使它们清晰而具体。我们只返回子代理的最终响应，因为主管不需要看到中间推理或工具调用。
## 创建主管agent
现在创建协调子代理的主管。主管只看到高级工具，并在域级别（而不是单个API级别）做出路由决策。

In [10]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
)

##  使用主管
现在用需要跨多个域协调的复杂请求测试您的完整系统：

In [ ]:
# 简单的丹玉请求
query = "Schedule a team standup for tomorrow at 9am"

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_976a752e44fc4de5bb781c)
 Call ID: call_976a752e44fc4de5bb781c
  Args:
    request: team standup tomorrow at 9am
================================= Tool Message =================================
Name: schedule_event

The team standup has been scheduled for tomorrow at 9:00 AM and will last for 15 minutes. The event is confirmed with the attendee(s).
================================== Ai Message ==================================

The team standup has been successfully scheduled for tomorrow at 9:00 AM. It will last for 15 minutes and is confirmed with all attendees.


主管将其标识为日历任务，调用schedule_event，日历代理处理日期解析和事件创建。

In [13]:
# 复杂的多域请求
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_72da0bee22de47f992d8fc)
 Call ID: call_72da0bee22de47f992d8fc
  Args:
    request: meeting with the design team next Tuesday at 2pm for 1 hour
  manage_email (call_a1a50a5c3e724412bcb27a)
 Call ID: call_a1a50a5c3e724412bcb27a
  Args:
    request: send them an email reminder about reviewing the new mockups
================================= Tool Message =================================
Name: manage_email

Could you please provide the recipient's email address, the subject line, and any specific details or message you'd like included in the reminder about reviewing the new mockups?
================================= Tool Message =================================
Name: schedule_event

I've scheduled your meeting with the design team for next Tuesday, April 4th, from 2:00 PM to 3:00 PM. The event has been successfully created.
================================== Ai Message ====

主管认识到这需要日历和电子邮件操作，为会议调用schedule_event，然后为提醒调用manage_email。每个子智能体完成自己的任务，监督器将两个结果综合成一个连贯的响应。

## 添加人在循环审查
谨慎的做法是对敏感行为进行“人在循环”审查。LangChain包括内置的中间件来检查工具调用，在本例中是由子代理调用的工具。

我们将create_calendar_event和send_email工具配置为中断，允许所有响应类型（批准、编辑、拒绝）

我们只向顶级代理添加检查指针。这是暂停和恢复执行所必需的。

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 


calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
    middleware=[ 
        HumanInTheLoopMiddleware( 
            interrupt_on={"create_calendar_event": True}, 
            description_prefix="Calendar event pending approval", 
        ), 
    ], 
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
    middleware=[ 
        HumanInTheLoopMiddleware( 
            interrupt_on={"send_email": True}, 
            description_prefix="Outbound email pending approval", 
        ), 
    ], 
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver(), 
)

让我们重复这个查询。请注意，我们将中断事件收集到一个列表中以访问下游：

In [15]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

config = {"configurable": {"thread_id": "6"}}

interrupts = []
for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    config,
):
    for update in step.values():
        if isinstance(update, dict):
            for message in update.get("messages", []):
                message.pretty_print()
        else:
            interrupt_ = update[0]
            interrupts.append(interrupt_)
            print(f"\nINTERRUPTED: {interrupt_.id}")

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_8edeb7a824754e4c87aa2e)
 Call ID: call_8edeb7a824754e4c87aa2e
  Args:
    request: meeting with the design team next Tuesday at 2pm for 1 hour
  manage_email (call_df2662c34c8a4e8bae5d1b)
 Call ID: call_df2662c34c8a4e8bae5d1b
  Args:
    request: send them an email reminder about reviewing the new mockups

INTERRUPTED: 77a66f64aebbb93a8ab325f3c9ddcf4f

INTERRUPTED: 9546e0d908356450940d905e7633842d


这次我们中断了执行。让我们来检查中断事件：

In [16]:
for interrupt_ in interrupts:
    for request in interrupt_.value["action_requests"]:
        print(f"INTERRUPTED: {interrupt_.id}")
        print(f"{request['description']}\n")

INTERRUPTED: 77a66f64aebbb93a8ab325f3c9ddcf4f
Calendar event pending approval

Tool: create_calendar_event
Args: {'title': 'Meeting with Design Team', 'start_time': '2023-04-04T14:00:00', 'end_time': '2023-04-04T15:00:00', 'attendees': ['design_team'], 'location': ''}

INTERRUPTED: 9546e0d908356450940d905e7633842d
Outbound email pending approval

Tool: send_email
Args: {'to': ['recipient@example.com'], 'subject': 'Reminder: Please Review the New Mockups', 'body': 'Hi,\n\nThis is a friendly reminder to review the new mockups when you get a chance.\n\nLet me know if you have any feedback or questions.\n\nBest regards,\n[Your Name]'}



我们可以通过使用命令引用每个中断的ID来指定每个中断的决策。更多细节请参考human-in-the-loop指南。出于演示目的，这里我们将接受日历事件，但编辑出站电子邮件的主题：

In [17]:
from langgraph.types import Command 

resume = {}
for interrupt_ in interrupts:
    if interrupt_.id == "2b56f299be313ad8bc689eff02973f16":
        # Edit email
        edited_action = interrupt_.value["action_requests"][0].copy()
        edited_action["arguments"]["subject"] = "Mockups reminder"
        resume[interrupt_.id] = {
            "decisions": [{"type": "edit", "edited_action": edited_action}]
        }
    else:
        resume[interrupt_.id] = {"decisions": [{"type": "approve"}]}

interrupts = []
for step in supervisor_agent.stream(
    Command(resume=resume), 
    config,
):
    for update in step.values():
        if isinstance(update, dict):
            for message in update.get("messages", []):
                message.pretty_print()
        else:
            interrupt_ = update[0]
            interrupts.append(interrupt_)
            print(f"\nINTERRUPTED: {interrupt_.id}")

================================= Tool Message =================================
Name: manage_email

The email reminder about reviewing the new mockups has been successfully sent to recipient@example.com with the subject: "Reminder: Please Review the New Mockups".
================================= Tool Message =================================
Name: schedule_event

The meeting with the design team has been scheduled for next Tuesday, April 4th, from 2:00 PM to 3:00 PM. Let me know if you need any adjustments!
================================== Ai Message ==================================

The meeting with the design team has been successfully scheduled for next Tuesday, April 4th, from 2:00 PM to 3:00 PM.  

Additionally, an email reminder has been sent to the team about reviewing the new mockups. Let me know if you'd like to make any changes or need further assistance!


## 进阶：控制信息流
默认情况下，子代理只接收来自上级的请求字符串。您可能希望传递其他上下文，例如对话历史记录或用户首选项。

In [ ]:
from langchain.tools import tool, ToolRuntime
# 将额外的对话内容传递给子agent
@tool
def schedule_event(
    request: str,
    runtime: ToolRuntime
) -> str:
    """Schedule calendar events using natural language."""
    # Customize context received by sub-agent
    original_user_message = next(
        message for message in runtime.state["messages"]
        if message.type == "human"#保存所有用户输入的消息
    )
    prompt = (
        "You are assisting with the following user inquiry:\n\n"
        f"{original_user_message.text}\n\n"
        "You are tasked with the following sub-request:\n\n"
        f"{request}"
    )
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": prompt}],
    })
    return result["messages"][-1].text

这允许子代理查看完整的会话上下文，这对于解决诸如“将其安排在明天同一时间”（引用之前的会话）之类的模糊性非常有用。

您还可以自定义返回给主管的信息：

In [19]:
import json

@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language."""
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })

    # Option 1: Return just the confirmation message
    return result["messages"][-1].text

    # Option 2: Return structured data
    # return json.dumps({
    #     "status": "success",
    #     "event_id": "evt_123",
    #     "summary": result["messages"][-1].text
    # })

确保子代理提示强调它们的最终消息应该包含所有相关信息。常见的故障模式是执行工具调用的子代理，但在其最终响应中不包括结果。
# 总结
何时使用监管者模式：

当您有多个不同的域（日历、电子邮件、CRM、数据库），每个域都有多个工具或复杂的逻辑，您需要集中的工作流控制，并且子代理不需要直接与用户交谈时，请使用主管模式。

对于只有几个工具的简单情况，请使用单个代理。当代理需要与用户进行对话时，请使用handoffs。对于代理之间的点对点协作，请考虑其他多代理模式。